# 04. Machine Learning Model Training & Imbalance Evaluation

## Project: Real-Time Credit Card Fraud Detection & Analytics System
**Objective**: Train and benchmark baseline Logistic Regression and Random Forest models with imbalance-aware loss weighting, evaluate PR-AUC / ROC-AUC, optimize decision thresholds, and quantify precision/recall trade-offs.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.models.train import train_fraud_detection_models
from src.models.evaluate import analyze_threshold_curve

results = train_fraud_detection_models(data_source='auto', save_artifacts=False)
baseline_metrics = results['baseline_metrics']
rf_metrics = results['main_metrics']

### 1. Comparative Model Performance Benchmark

In [ ]:
metrics_comparison = pd.DataFrame([
    {
        'Model': 'Logistic Regression (Baseline)',
        'Accuracy': f"{baseline_metrics['accuracy']:.4f}",
        'Precision': f"{baseline_metrics['precision']:.4f}",
        'Recall': f"{baseline_metrics['recall']:.4f}",
        'F1 Score': f"{baseline_metrics['f1']:.4f}",
        'ROC-AUC': f"{baseline_metrics['roc_auc']:.4f}",
        'PR-AUC': f"{baseline_metrics['pr_auc']:.4f}",
        'False Positives': baseline_metrics['false_positives'],
        'False Negatives': baseline_metrics['false_negatives']
    },
    {
        'Model': 'Random Forest (Main)',
        'Accuracy': f"{rf_metrics['accuracy']:.4f}",
        'Precision': f"{rf_metrics['precision']:.4f}",
        'Recall': f"{rf_metrics['recall']:.4f}",
        'F1 Score': f"{rf_metrics['f1']:.4f}",
        'ROC-AUC': f"{rf_metrics['roc_auc']:.4f}",
        'PR-AUC': f"{rf_metrics['pr_auc']:.4f}",
        'False Positives': rf_metrics['false_positives'],
        'False Negatives': rf_metrics['false_negatives']
    }
])
print(metrics_comparison.to_string(index=False))

### 2. Confusion Matrices Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(baseline_metrics['confusion_matrix'], annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred Legit', 'Pred Fraud'], yticklabels=['True Legit', 'True Fraud'])
axes[0].set_title('Logistic Regression Confusion Matrix')

sns.heatmap(rf_metrics['confusion_matrix'], annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Pred Legit', 'Pred Fraud'], yticklabels=['True Legit', 'True Fraud'])
axes[1].set_title('Random Forest Confusion Matrix')
plt.tight_layout()
plt.show()

### 3. Feature Importance Analysis (Top Discriminators)

In [ ]:
imp_df = pd.DataFrame(results['feature_importances'][:12])
plt.figure(figsize=(10, 5))
sns.barplot(data=imp_df, x='importance', y='feature', palette='viridis')
plt.title('Top Predictive Features in Random Forest Model', fontsize=13)
plt.xlabel('Gini Importance')
plt.show()